# X-ray preprocessing exploration

Companion to `src/preprocess.py`. Use it to:
- Inventory which frames have real data
- Tune percentile / CLAHE / gamma values visually
- Compare single frames vs the mean stack
- Make figures

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from preprocess import (
    load_tiff, find_usable_frames,
    flat_field, percentile_normalize, apply_clahe, apply_gamma,
    process_one,
    PCT_LOW, PCT_HIGH, CLAHE_CLIP, GAMMA,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load and inventory

In [ ]:
OBJECT_PATH = '../../data/object1_hole.tiff'
DARK_PATH   = '../../data/dark.tiff'
FLAT_PATH   = None

obj = load_tiff(OBJECT_PATH)
print(f'object: {obj.shape}, dtype={obj.dtype}')

for i in range(len(obj)):
    p = obj[i]
    nz = 100 * np.count_nonzero(p) / p.size
    tag = 'real' if nz > 50 else ('partial' if nz > 1 else 'empty')
    print(f'  page {i:2d}: max={p.max():5d}, mean={p.mean():6.1f}, nonzero={nz:5.1f}%  ({tag})')

In [ ]:
dark = load_tiff(DARK_PATH) if DARK_PATH else None
if dark is not None:
    print(f'dark: {dark.shape}, all-zero={bool((dark==0).all())}, '
          f'nonzero pixels={np.count_nonzero(dark)} / {dark.size}')

## 2. Usable frames + mean stack

In [ ]:
frames, kept = find_usable_frames(obj)
mean_frame = frames.mean(axis=0)
print(f'kept {len(kept)} frames: {kept}')
print(f'mean: min={mean_frame.min():.1f}, max={mean_frame.max():.1f}, mean={mean_frame.mean():.1f}')

# SNR check on a flat-ish patch. Mean stack std should be ~sqrt(N)x lower.
roi = (slice(100, 200), slice(200, 400))
print(f'\nROI std, single frame: {frames[0][roi].std():.2f}')
print(f'ROI std, mean stack:   {mean_frame[roi].std():.2f}')
print(f'Expected reduction: ~{np.sqrt(len(frames)):.2f}x')

## 3. Run all stages and view

In [ ]:
stages = process_one(mean_frame, dark=dark)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, name in zip(axes.flat, ['raw', 'normalized', 'clahe', 'gamma']):
    im = ax.imshow(stages[name], cmap='gray')
    ax.set_title(name)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f'mean stack | pct=[{PCT_LOW},{PCT_HIGH}] CLAHE={CLAHE_CLIP} gamma={GAMMA}')
fig.tight_layout();

## 4. CLAHE clip-limit sweep

In [ ]:
norm = stages['normalized']
clips = [0.005, 0.01, 0.02, 0.04]

fig, axes = plt.subplots(1, len(clips), figsize=(5*len(clips), 5))
for ax, c in zip(axes, clips):
    ax.imshow(apply_clahe(norm, clip_limit=c), cmap='gray')
    ax.set_title(f'clip={c}')
    ax.axis('off')
fig.tight_layout();

## 5. Single frame vs mean stack

In [ ]:
single = percentile_normalize(frames[0])
stacked = percentile_normalize(mean_frame)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(single, cmap='gray');  axes[0].set_title('Frame 0')
axes[1].imshow(stacked, cmap='gray'); axes[1].set_title(f'Mean of {len(frames)}')
for ax in axes: ax.axis('off')
fig.tight_layout();

## Scratch